
# TP3 - Pequeña red para clasificación
**Autor:** Agustin Lara  
**Base:** continuación del TP2 adaptada al problema de clasificación multiclase.

## Objetivo
Resolver el problema bidimensional de clasificación en **tres categorías excluyentes** (Rojo, Azul y Amarillo) usando una red neuronal pequeña con:
- **2 inputs**: $(x, y)$
- **3 neuronas de salida**
- **softmax** en la última capa
- **one-hot encoding** para las clases
- **entropía cruzada multiclase** para entrenar la red

Además, se analiza:
1. efecto del **learning rate**
2. efecto del **número de épocas**
3. efecto de la **arquitectura**
4. efecto de la **cantidad/calidad de datos**
5. comparación entre **full-batch** y **mini-batch training**
6. comparación entre **sigmoide** y **ReLU** en capas ocultas

> Nota: el PDF mezcla la idea de *softmax + entropía cruzada* con una fórmula escrita como BCE. Como acá el problema es **multiclase excluyente**, uso la formulación estándar y consistente: **softmax + categorical cross-entropy** con targets one-hot.


In [ ]:

import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from IPython.display import display

np.random.seed(42)

RESULTS_DIR = Path("results_tp3")
RESULTS_DIR.mkdir(exist_ok=True)

CLASS_NAMES = ["Rojo", "Azul", "Amarillo"]
CLASS_COLORS = ["red", "blue", "gold"]
CLASS_TO_INT = {name: i for i, name in enumerate(CLASS_NAMES)}

print("Results dir:", RESULTS_DIR.resolve())



## 1. Generación del dataset 2D pedido por la consigna
La región de clases sigue la figura del TP:
- **Amarillo** arriba de las rectas inclinadas
- **Rojo** abajo a la izquierda
- **Azul** abajo a la derecha
- la triple frontera queda en **(0, 0)**

Una forma simple de codificar la geometría es:
- Amarillo si $y \ge 0.5\,|x|$
- Si no, Rojo si $x < 0$
- Si no, Azul


In [ ]:

def generate_tp3_grid(step=0.10):
    """
    Genera una grilla (x, y) en [-1, 1] x [-1, 1] y asigna la clase correcta.

    Regla geométrica inspirada en la figura del enunciado:
      - Amarillo: y >= 0.5 * |x|
      - Rojo    : y < 0.5 * |x| y x < 0
      - Azul    : y < 0.5 * |x| y x >= 0
    """
    xs = np.arange(-1.0, 1.0 + 1e-9, step)
    ys = np.arange(-1.0, 1.0 + 1e-9, step)
    XX, YY = np.meshgrid(xs, ys)
    X = np.column_stack([XX.ravel(), YY.ravel()])

    y_idx = []
    for x, y in X:
        if y >= 0.5 * abs(x):
            y_idx.append(CLASS_TO_INT["Amarillo"])
        elif x < 0:
            y_idx.append(CLASS_TO_INT["Rojo"])
        else:
            y_idx.append(CLASS_TO_INT["Azul"])

    y_idx = np.array(y_idx, dtype=int)
    y_onehot = np.eye(3)[y_idx]

    df = pd.DataFrame({
        "x": X[:, 0],
        "y": X[:, 1],
        "class_idx": y_idx,
        "class_name": [CLASS_NAMES[i] for i in y_idx]
    })
    return X, y_idx, y_onehot, xs, ys, df


X_demo, y_demo_idx, y_demo_onehot, xs_demo, ys_demo, df_demo = generate_tp3_grid(step=0.10)
print("Cantidad de puntos:", len(X_demo))
print(df_demo.head())
print("\nDistribución de clases:")
print(df_demo["class_name"].value_counts())


In [ ]:

plt.figure(figsize=(7, 6))
for class_idx, class_name in enumerate(CLASS_NAMES):
    mask = y_demo_idx == class_idx
    plt.scatter(
        X_demo[mask, 0],
        X_demo[mask, 1],
        s=35,
        c=CLASS_COLORS[class_idx],
        label=class_name,
        edgecolor="none"
    )

plt.axhline(0, color="black", linewidth=0.8, alpha=0.3)
plt.axvline(0, color="black", linewidth=0.8, alpha=0.3)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Dataset TP3: clasificación correcta sobre la grilla")
plt.legend()
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "01_dataset_tp3.png", dpi=300, bbox_inches="tight")
plt.show()



## 2. Red neuronal para clasificación multiclase
Ahora la red cambia respecto del TP2:
- la entrada tiene **2 variables**
- la salida tiene **3 neuronas**
- la última capa aplica **softmax** para obtener probabilidades que suman 1


In [ ]:

def sigmoid(x):
    x = np.clip(x, -50, 50)
    return 1 / (1 + np.exp(-x))


def sigmoid_derivative_from_z(z):
    a = sigmoid(z)
    return a * (1 - a)


def relu(x):
    return np.maximum(0, x)


def relu_derivative_from_z(z):
    return (z > 0).astype(float)


def softmax(x):
    x = x - np.max(x, axis=1, keepdims=True)  # estabilidad numérica
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)


def initialize_network_classifier(input_size, hidden_sizes, output_size, hidden_activation="sigmoid", seed=42):
    """
    Inicializa una red como lista de [weights, biases].
    weights shape: (n_neurons_next, n_neurons_prev)
    biases shape : (n_neurons_next,)
    """
    rng = np.random.default_rng(seed)
    layer_sizes = [input_size] + hidden_sizes + [output_size]

    network = []
    for i in range(len(layer_sizes) - 1):
        fan_in = layer_sizes[i]
        fan_out = layer_sizes[i + 1]

        # salida con Xavier; ocultas con Xavier o He según activación
        if i == len(layer_sizes) - 2:
            limit = np.sqrt(6 / (fan_in + fan_out))
            weights = rng.uniform(-limit, limit, size=(fan_out, fan_in))
        else:
            if hidden_activation == "relu":
                std = np.sqrt(2 / fan_in)
                weights = rng.normal(0, std, size=(fan_out, fan_in))
            else:
                limit = np.sqrt(6 / (fan_in + fan_out))
                weights = rng.uniform(-limit, limit, size=(fan_out, fan_in))

        biases = np.zeros(fan_out)
        network.append([weights, biases])

    return network


def forward_layer_linear(inputs, weights, biases):
    return np.dot(inputs, weights.T) + biases


def forward_network_classifier(inputs, network, hidden_activation="sigmoid"):
    """
    Forward pass completo.
    Devuelve activaciones y preactivaciones (z) para usar en backprop.
    """
    activations = [inputs]
    pre_activations = []
    a = inputs

    for i, (weights, biases) in enumerate(network):
        z = forward_layer_linear(a, weights, biases)
        pre_activations.append(z)

        if i == len(network) - 1:
            a = softmax(z)
        else:
            a = relu(z) if hidden_activation == "relu" else sigmoid(z)

        activations.append(a)

    return activations, pre_activations


def forward_network_without_softmax(inputs, network, hidden_activation="sigmoid"):
    """
    Igual que el forward normal, pero deja la última capa como logits sin softmax.
    Se usa solo para comparar outputs con/sin softmax.
    """
    a = inputs.copy()
    for i, (weights, biases) in enumerate(network):
        z = forward_layer_linear(a, weights, biases)
        if i == len(network) - 1:
            a = z
        else:
            a = relu(z) if hidden_activation == "relu" else sigmoid(z)
    return a


In [ ]:

# Comparación pedida en el TP: output con y sin softmax
network_test = initialize_network_classifier(
    input_size=2,
    hidden_sizes=[4, 4],
    output_size=3,
    hidden_activation="sigmoid",
    seed=42
)

sample_inputs = np.array([
    [-0.8, -0.8],
    [-0.4,  0.8],
    [ 0.7, -0.6],
    [ 0.2,  0.9],
])

logits = forward_network_without_softmax(sample_inputs, network_test, hidden_activation="sigmoid")
probs = forward_network_classifier(sample_inputs, network_test, hidden_activation="sigmoid")[0][-1]

comparison_df = pd.DataFrame(
    np.hstack([sample_inputs, logits, probs, probs.sum(axis=1, keepdims=True)]),
    columns=[
        "x", "y",
        "logit_rojo", "logit_azul", "logit_amarillo",
        "p_rojo", "p_azul", "p_amarillo",
        "sum_probs"
    ]
)

display(comparison_df.round(4))
print("Observación: los logits no están normalizados; después de softmax las probabilidades suman 1.")



## 3. Loss, accuracy, matriz de confusión y backpropagation
Para entrenar la red usamos:
- **entropía cruzada multiclase** con targets one-hot
- **accuracy** como porcentaje de aciertos
- **matriz de confusión** para ver en qué clases se equivoca


In [ ]:

def cross_entropy_loss(y_pred, y_true):
    eps = 1e-12
    return -np.mean(np.sum(y_true * np.log(y_pred + eps), axis=1))


def accuracy_score_np(y_true_idx, y_pred_idx):
    return np.mean(y_true_idx == y_pred_idx)


def confusion_matrix_np(y_true_idx, y_pred_idx, n_classes=3):
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for t, p in zip(y_true_idx, y_pred_idx):
        cm[t, p] += 1
    return cm


def backward_network_classifier(activations, pre_activations, network, y_true, learning_rate, hidden_activation="sigmoid"):
    """
    Backpropagation para softmax + cross-entropy.
    """
    y_pred = activations[-1]
    batch_size = y_true.shape[0]
    loss = cross_entropy_loss(y_pred, y_true)

    # Gradiente de CE respecto de logits de salida con softmax
    delta = (y_pred - y_true) / batch_size

    for i in reversed(range(len(network))):
        weights, biases = network[i]
        inputs = activations[i]

        d_weights = np.dot(delta.T, inputs)
        d_biases = np.sum(delta, axis=0)

        old_weights = weights.copy()
        network[i][0] = weights - learning_rate * d_weights
        network[i][1] = biases - learning_rate * d_biases

        if i > 0:
            d_inputs = np.dot(delta, old_weights)
            z_prev = pre_activations[i - 1]
            if hidden_activation == "relu":
                delta = d_inputs * relu_derivative_from_z(z_prev)
            else:
                delta = d_inputs * sigmoid_derivative_from_z(z_prev)

    return loss


def train_network_classifier(
    X,
    y_idx,
    y_onehot,
    input_size,
    hidden_sizes,
    output_size,
    epochs=500,
    learning_rate=0.5,
    hidden_activation="sigmoid",
    seed=42,
    batch_size=None,
    shuffle=True,
    verbose_every=100,
):
    network = initialize_network_classifier(
        input_size=input_size,
        hidden_sizes=hidden_sizes,
        output_size=output_size,
        hidden_activation=hidden_activation,
        seed=seed,
    )

    n_samples = len(X)
    if batch_size is None:
        batch_size = n_samples

    rng = np.random.default_rng(seed)
    history = {
        "loss": [],
        "accuracy": [],
        "epoch_time_s": []
    }

    for epoch in range(1, epochs + 1):
        t0 = time.perf_counter()

        if shuffle:
            order = rng.permutation(n_samples)
            X_epoch = X[order]
            y_idx_epoch = y_idx[order]
            y_onehot_epoch = y_onehot[order]
        else:
            X_epoch = X
            y_idx_epoch = y_idx
            y_onehot_epoch = y_onehot

        epoch_loss_accumulator = 0.0

        for start in range(0, n_samples, batch_size):
            stop = min(start + batch_size, n_samples)
            X_batch = X_epoch[start:stop]
            y_batch = y_onehot_epoch[start:stop]

            activations, pre_activations = forward_network_classifier(
                X_batch,
                network,
                hidden_activation=hidden_activation
            )

            batch_loss = backward_network_classifier(
                activations,
                pre_activations,
                network,
                y_batch,
                learning_rate=learning_rate,
                hidden_activation=hidden_activation
            )

            epoch_loss_accumulator += batch_loss * len(X_batch)

        epoch_loss = epoch_loss_accumulator / n_samples

        y_pred_epoch = forward_network_classifier(X, network, hidden_activation=hidden_activation)[0][-1]
        y_pred_idx_epoch = np.argmax(y_pred_epoch, axis=1)
        epoch_acc = accuracy_score_np(y_idx, y_pred_idx_epoch)

        history["loss"].append(epoch_loss)
        history["accuracy"].append(epoch_acc)
        history["epoch_time_s"].append(time.perf_counter() - t0)

        if verbose_every is not None and (epoch == 1 or epoch % verbose_every == 0 or epoch == epochs):
            print(
                f"Epoch {epoch:4d}/{epochs} | loss={epoch_loss:.4f} | accuracy={epoch_acc:.4f}"
            )

    return network, history


In [ ]:

def evaluate_network(network, X, y_idx, y_onehot, hidden_activation="sigmoid"):
    y_pred = forward_network_classifier(X, network, hidden_activation=hidden_activation)[0][-1]
    y_pred_idx = np.argmax(y_pred, axis=1)

    metrics = {
        "loss": cross_entropy_loss(y_pred, y_onehot),
        "accuracy": accuracy_score_np(y_idx, y_pred_idx),
        "confusion_matrix": confusion_matrix_np(y_idx, y_pred_idx, n_classes=3),
        "y_pred": y_pred,
        "y_pred_idx": y_pred_idx,
    }
    return metrics


def plot_training_history(history, title_prefix="Training"):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history["loss"], linewidth=2)
    axes[0].set_title(f"{title_prefix} - Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Cross-entropy")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(history["accuracy"], linewidth=2)
    axes[1].set_title(f"{title_prefix} - Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    return fig


def plot_confusion_matrix(cm, class_names, title="Confusion matrix"):
    fig, ax = plt.subplots(figsize=(5.5, 5))
    im = ax.imshow(cm, cmap="Blues")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))
    ax.set_xticklabels(class_names)
    ax.set_yticklabels(class_names)
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Clase real")
    ax.set_title(title)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="black")

    plt.tight_layout()
    return fig


def plot_decision_map(X, y_true_idx, y_pred_idx, title="Mapa de decisión"):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
    cmap = ListedColormap(CLASS_COLORS)

    axes[0].scatter(X[:, 0], X[:, 1], c=y_true_idx, cmap=cmap, s=14, edgecolor="none")
    axes[0].set_title("Clases reales")
    axes[0].set_xlabel("x")
    axes[0].set_ylabel("y")
    axes[0].grid(True, alpha=0.2)

    axes[1].scatter(X[:, 0], X[:, 1], c=y_pred_idx, cmap=cmap, s=14, edgecolor="none")
    axes[1].set_title("Clases predichas")
    axes[1].set_xlabel("x")
    axes[1].grid(True, alpha=0.2)

    fig.suptitle(title)
    plt.tight_layout()
    return fig


def plot_probability_surfaces(X, y_true_idx, y_pred_prob, title="Probabilidades por clase"):
    fig = plt.figure(figsize=(16, 4.8))

    for class_idx, class_name in enumerate(CLASS_NAMES):
        ax = fig.add_subplot(1, 3, class_idx + 1, projection="3d")

        # plano base con clase real en z=0
        ax.scatter(
            X[:, 0], X[:, 1], np.zeros(len(X)),
            c=[CLASS_COLORS[i] for i in y_true_idx],
            s=8, alpha=0.25, depthshade=False
        )

        # probabilidad predicha de la clase class_idx
        ax.scatter(
            X[:, 0], X[:, 1], y_pred_prob[:, class_idx],
            c=CLASS_COLORS[class_idx],
            s=10, alpha=0.85, depthshade=False
        )

        ax.set_title(f"Probabilidad de {class_name}")
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.set_zlabel("p")
        ax.set_zlim(0, 1.05)
        ax.view_init(elev=24, azim=-60)

    fig.suptitle(title)
    plt.tight_layout()
    return fig



## 4. Ejercicio 3.1 - Entrenamiento y evaluación de la red
Voy a usar:
- un **dataset de entrenamiento** con `step = 0.10`
- un **dataset de evaluación** más fino con `step = 0.025`

Esto permite medir capacidad de generalización sobre una grilla mucho más densa que la usada para entrenar.


In [ ]:

X_train, y_train_idx, y_train_onehot, xs_train, ys_train, df_train = generate_tp3_grid(step=0.10)
X_eval, y_eval_idx, y_eval_onehot, xs_eval, ys_eval, df_eval = generate_tp3_grid(step=0.025)

print("Training set:", X_train.shape, "| step=0.10")
print("Evaluation set:", X_eval.shape, "| step=0.025")


In [ ]:

# Modelo base con activación sigmoidea en ocultas
baseline_sigmoid_network, baseline_sigmoid_history = train_network_classifier(
    X=X_train,
    y_idx=y_train_idx,
    y_onehot=y_train_onehot,
    input_size=2,
    hidden_sizes=[8, 8],
    output_size=3,
    epochs=1000,
    learning_rate=0.5,
    hidden_activation="sigmoid",
    seed=42,
    batch_size=None,
    verbose_every=200,
)

baseline_sigmoid_metrics = evaluate_network(
    baseline_sigmoid_network,
    X_eval,
    y_eval_idx,
    y_eval_onehot,
    hidden_activation="sigmoid"
)

print("\n=== Resultado baseline sigmoidea ===")
print(f"Loss final (eval): {baseline_sigmoid_metrics['loss']:.4f}")
print(f"Accuracy (eval) : {baseline_sigmoid_metrics['accuracy']:.4f}")
print("Confusion matrix:")
print(baseline_sigmoid_metrics["confusion_matrix"])


In [ ]:

fig = plot_training_history(baseline_sigmoid_history, title_prefix="Baseline sigmoid")
fig.savefig(RESULTS_DIR / "02_baseline_sigmoid_training_history.png", dpi=300, bbox_inches="tight")
plt.show()

fig = plot_confusion_matrix(
    baseline_sigmoid_metrics["confusion_matrix"],
    CLASS_NAMES,
    title="Baseline sigmoid - confusion matrix"
)
fig.savefig(RESULTS_DIR / "03_baseline_sigmoid_confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

fig = plot_decision_map(
    X_eval,
    y_eval_idx,
    baseline_sigmoid_metrics["y_pred_idx"],
    title="Baseline sigmoid - clases reales vs predichas"
)
fig.savefig(RESULTS_DIR / "04_baseline_sigmoid_decision_map.png", dpi=300, bbox_inches="tight")
plt.show()

fig = plot_probability_surfaces(
    X_eval,
    y_eval_idx,
    baseline_sigmoid_metrics["y_pred"],
    title="Baseline sigmoid - superficies de probabilidad"
)
fig.savefig(RESULTS_DIR / "05_baseline_sigmoid_probability_surfaces.png", dpi=300, bbox_inches="tight")
plt.show()



### 4.1 Efecto del número de épocas


In [ ]:

def run_experiment(hidden_sizes, epochs, learning_rate, hidden_activation, step_train=0.10, batch_size=None, seed=42):
    X_tr, y_tr_idx, y_tr_oh, *_ = generate_tp3_grid(step=step_train)
    X_ev, y_ev_idx, y_ev_oh, *_ = generate_tp3_grid(step=0.025)

    t0 = time.perf_counter()
    network, history = train_network_classifier(
        X=X_tr,
        y_idx=y_tr_idx,
        y_onehot=y_tr_oh,
        input_size=2,
        hidden_sizes=hidden_sizes,
        output_size=3,
        epochs=epochs,
        learning_rate=learning_rate,
        hidden_activation=hidden_activation,
        seed=seed,
        batch_size=batch_size,
        verbose_every=None,
    )
    total_time = time.perf_counter() - t0

    metrics = evaluate_network(network, X_ev, y_ev_idx, y_ev_oh, hidden_activation=hidden_activation)

    result = {
        "architecture": str(hidden_sizes),
        "epochs": epochs,
        "learning_rate": learning_rate,
        "activation": hidden_activation,
        "step_train": step_train,
        "n_train": len(X_tr),
        "batch_size": len(X_tr) if batch_size is None else batch_size,
        "train_time_s": total_time,
        "eval_loss": metrics["loss"],
        "eval_accuracy": metrics["accuracy"],
    }
    return result, history, network, metrics


epoch_values = [100, 300, 500, 1000, 2000]
epoch_results = []

for e in epoch_values:
    result, _, _, _ = run_experiment(
        hidden_sizes=[8, 8],
        epochs=e,
        learning_rate=0.5,
        hidden_activation="sigmoid",
        step_train=0.10,
        batch_size=None,
        seed=42,
    )
    epoch_results.append(result)


df_epochs = pd.DataFrame(epoch_results)
display(df_epochs.round(4))
df_epochs.to_csv(RESULTS_DIR / "06_epochs_summary.csv", index=False)


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(df_epochs["epochs"], df_epochs["eval_loss"], marker="o", linewidth=2)
axes[0].set_xlabel("Epochs")
axes[0].set_ylabel("Eval loss")
axes[0].set_title("Efecto del número de épocas")
axes[0].grid(True, alpha=0.3)

axes[1].plot(df_epochs["epochs"], df_epochs["eval_accuracy"], marker="o", linewidth=2)
axes[1].set_xlabel("Epochs")
axes[1].set_ylabel("Eval accuracy")
axes[1].set_title("Accuracy vs epochs")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "07_epochs_effect.png", dpi=300, bbox_inches="tight")
plt.show()



### 4.2 Efecto del learning rate


In [ ]:

lr_values = [0.01, 0.05, 0.10, 0.30, 0.50, 1.00]
lr_results = []

for lr in lr_values:
    result, _, _, _ = run_experiment(
        hidden_sizes=[8, 8],
        epochs=500,
        learning_rate=lr,
        hidden_activation="sigmoid",
        step_train=0.10,
        batch_size=None,
        seed=42,
    )
    lr_results.append(result)


df_lr = pd.DataFrame(lr_results)
display(df_lr.round(4))
df_lr.to_csv(RESULTS_DIR / "08_learning_rate_summary.csv", index=False)


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(df_lr["learning_rate"], df_lr["eval_loss"], marker="o", linewidth=2)
axes[0].set_xlabel("Learning rate")
axes[0].set_ylabel("Eval loss")
axes[0].set_title("Efecto del learning rate")
axes[0].grid(True, alpha=0.3)

axes[1].plot(df_lr["learning_rate"], df_lr["eval_accuracy"], marker="o", linewidth=2)
axes[1].set_xlabel("Learning rate")
axes[1].set_ylabel("Eval accuracy")
axes[1].set_title("Accuracy vs learning rate")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "09_learning_rate_effect.png", dpi=300, bbox_inches="tight")
plt.show()



### 4.3 Efecto de la arquitectura


In [ ]:

architectures = [
    [3],
    [5],
    [3, 3],
    [5, 5],
    [8, 8],
    [8, 8, 8],
    [12, 8, 4],
]

arch_results = []
for arch in architectures:
    result, _, _, _ = run_experiment(
        hidden_sizes=arch,
        epochs=500,
        learning_rate=0.5,
        hidden_activation="sigmoid",
        step_train=0.10,
        batch_size=None,
        seed=42,
    )
    arch_results.append(result)


df_arch = pd.DataFrame(arch_results)
display(df_arch.round(4))
df_arch.to_csv(RESULTS_DIR / "10_architecture_summary_sigmoid.csv", index=False)


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(df_arch["architecture"], df_arch["eval_loss"])
axes[0].set_xlabel("Architecture")
axes[0].set_ylabel("Eval loss")
axes[0].set_title("Arquitectura vs loss")
axes[0].grid(True, axis="y", alpha=0.3)

axes[1].bar(df_arch["architecture"], df_arch["eval_accuracy"])
axes[1].set_xlabel("Architecture")
axes[1].set_ylabel("Eval accuracy")
axes[1].set_title("Arquitectura vs accuracy")
axes[1].grid(True, axis="y", alpha=0.3)

for ax in axes:
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "11_architecture_effect_sigmoid.png", dpi=300, bbox_inches="tight")
plt.show()



### 4.4 Efecto de la cantidad/calidad de datos
Como el input es bidimensional, la cantidad de puntos crece aproximadamente como $n^2$ al refinar la grilla.

Acá uso el **step** como control de cantidad/calidad de datos:
- **step grande**: pocos puntos, muestreo grueso
- **step chico**: muchos puntos, muestreo más fino


In [ ]:

step_values = [0.40, 0.20, 0.10, 0.05]
data_results = []

for step in step_values:
    result, _, _, _ = run_experiment(
        hidden_sizes=[5, 5],
        epochs=500,
        learning_rate=0.5,
        hidden_activation="sigmoid",
        step_train=step,
        batch_size=None,
        seed=42,
    )
    data_results.append(result)


df_data = pd.DataFrame(data_results)
display(df_data.round(4))
df_data.to_csv(RESULTS_DIR / "12_data_size_summary.csv", index=False)


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(df_data["n_train"], df_data["train_time_s"], marker="o", linewidth=2)
axes[0].set_xlabel("Cantidad de puntos de entrenamiento")
axes[0].set_ylabel("Tiempo de entrenamiento (s)")
axes[0].set_title("Tiempo vs cantidad de datos")
axes[0].grid(True, alpha=0.3)

axes[1].plot(df_data["n_train"], df_data["eval_accuracy"], marker="o", linewidth=2)
axes[1].set_xlabel("Cantidad de puntos de entrenamiento")
axes[1].set_ylabel("Eval accuracy")
axes[1].set_title("Accuracy vs cantidad de datos")
axes[1].grid(True, alpha=0.3)

axes[2].plot(df_data["n_train"], df_data["eval_loss"], marker="o", linewidth=2)
axes[2].set_xlabel("Cantidad de puntos de entrenamiento")
axes[2].set_ylabel("Eval loss")
axes[2].set_title("Loss vs cantidad de datos")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "13_data_size_effect.png", dpi=300, bbox_inches="tight")
plt.show()



## 5. Ejercicio 3.2 - Mini-batch training
Ahora se compara el entrenamiento:
- **full-batch**: actualiza pesos una vez por época usando todos los datos
- **mini-batch**: actualiza varias veces dentro de cada época usando lotes pequeños


In [ ]:

full_batch_result, full_batch_history, full_batch_network, full_batch_metrics = run_experiment(
    hidden_sizes=[5, 5],
    epochs=200,
    learning_rate=0.5,
    hidden_activation="sigmoid",
    step_train=0.10,
    batch_size=None,
    seed=42,
)

mini_batch_result, mini_batch_history, mini_batch_network, mini_batch_metrics = run_experiment(
    hidden_sizes=[5, 5],
    epochs=200,
    learning_rate=0.5,
    hidden_activation="sigmoid",
    step_train=0.10,
    batch_size=64,
    seed=42,
)

comparison_batch = pd.DataFrame([
    {
        "mode": "full-batch",
        **full_batch_result,
    },
    {
        "mode": "mini-batch (64)",
        **mini_batch_result,
    },
])

display(comparison_batch[["mode", "epochs", "batch_size", "train_time_s", "eval_loss", "eval_accuracy"]].round(4))
comparison_batch.to_csv(RESULTS_DIR / "14_full_vs_minibatch_summary.csv", index=False)


In [ ]:

# Época en la que se alcanza 95% de accuracy de entrenamiento
full_arr = np.array(full_batch_history["accuracy"])
mini_arr = np.array(mini_batch_history["accuracy"])

full_reach = np.where(full_arr >= 0.95)[0]
mini_reach = np.where(mini_arr >= 0.95)[0]

print("Full-batch alcanza 95% de accuracy en época:", None if len(full_reach) == 0 else int(full_reach[0] + 1))
print("Mini-batch alcanza 95% de accuracy en época:", None if len(mini_reach) == 0 else int(mini_reach[0] + 1))


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(full_batch_history["loss"], label="Full-batch", linewidth=2)
axes[0].plot(mini_batch_history["loss"], label="Mini-batch (64)", linewidth=2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Full-batch vs mini-batch: loss")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(full_batch_history["accuracy"], label="Full-batch", linewidth=2)
axes[1].plot(mini_batch_history["accuracy"], label="Mini-batch (64)", linewidth=2)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Full-batch vs mini-batch: accuracy")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / "15_full_vs_minibatch_curves.png", dpi=300, bbox_inches="tight")
plt.show()



## 6. Ejercicio 3.3 - ReLU en lugar de sigmoide
Se reemplaza la activación oculta por **ReLU** y se compara:
- velocidad de entrenamiento
- resultado final
- efecto de distintas arquitecturas


In [ ]:

sigmoid_result, sigmoid_history, sigmoid_network, sigmoid_metrics = run_experiment(
    hidden_sizes=[8, 8],
    epochs=300,
    learning_rate=0.5,
    hidden_activation="sigmoid",
    step_train=0.10,
    batch_size=None,
    seed=42,
)

relu_result, relu_history, relu_network, relu_metrics = run_experiment(
    hidden_sizes=[8, 8],
    epochs=300,
    learning_rate=0.10,
    hidden_activation="relu",
    step_train=0.10,
    batch_size=None,
    seed=42,
)

activation_comparison = pd.DataFrame([
    {"activation": "sigmoid", **sigmoid_result},
    {"activation": "relu", **relu_result},
])

display(activation_comparison[["activation", "epochs", "learning_rate", "train_time_s", "eval_loss", "eval_accuracy"]].round(4))
activation_comparison.to_csv(RESULTS_DIR / "16_sigmoid_vs_relu_summary.csv", index=False)


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(sigmoid_history["loss"], label="Sigmoid", linewidth=2)
axes[0].plot(relu_history["loss"], label="ReLU", linewidth=2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Sigmoid vs ReLU - loss")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(sigmoid_history["accuracy"], label="Sigmoid", linewidth=2)
axes[1].plot(relu_history["accuracy"], label="ReLU", linewidth=2)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Sigmoid vs ReLU - accuracy")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / "17_sigmoid_vs_relu_curves.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:

fig = plot_confusion_matrix(relu_metrics["confusion_matrix"], CLASS_NAMES, title="ReLU - confusion matrix")
fig.savefig(RESULTS_DIR / "18_relu_confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

fig = plot_decision_map(X_eval, y_eval_idx, relu_metrics["y_pred_idx"], title="ReLU - clases reales vs predichas")
fig.savefig(RESULTS_DIR / "19_relu_decision_map.png", dpi=300, bbox_inches="tight")
plt.show()

fig = plot_probability_surfaces(X_eval, y_eval_idx, relu_metrics["y_pred"], title="ReLU - superficies de probabilidad")
fig.savefig(RESULTS_DIR / "20_relu_probability_surfaces.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:

relu_architectures = [
    [3],
    [5],
    [3, 3],
    [5, 5],
    [8, 8],
    [8, 8, 8],
    [12, 8, 4],
]

relu_arch_results = []
for arch in relu_architectures:
    result, _, _, _ = run_experiment(
        hidden_sizes=arch,
        epochs=300,
        learning_rate=0.10,
        hidden_activation="relu",
        step_train=0.10,
        batch_size=None,
        seed=42,
    )
    relu_arch_results.append(result)


df_relu_arch = pd.DataFrame(relu_arch_results)
display(df_relu_arch.round(4))
df_relu_arch.to_csv(RESULTS_DIR / "21_architecture_summary_relu.csv", index=False)


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(df_relu_arch["architecture"], df_relu_arch["eval_loss"])
axes[0].set_xlabel("Architecture")
axes[0].set_ylabel("Eval loss")
axes[0].set_title("ReLU: arquitectura vs loss")
axes[0].grid(True, axis="y", alpha=0.3)

axes[1].bar(df_relu_arch["architecture"], df_relu_arch["eval_accuracy"])
axes[1].set_xlabel("Architecture")
axes[1].set_ylabel("Eval accuracy")
axes[1].set_title("ReLU: arquitectura vs accuracy")
axes[1].grid(True, axis="y", alpha=0.3)

for ax in axes:
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "22_architecture_effect_relu.png", dpi=300, bbox_inches="tight")
plt.show()



## 7. Conclusiones


In [ ]:

summary_lines = [
    "Conclusiones generales del TP3:",
    "",
    f"1. La red multiclase con softmax resolvió correctamente el problema 2D. En la baseline con sigmoide se obtuvo accuracy de evaluación = {baseline_sigmoid_metrics['accuracy']:.4f}.",
    f"2. Aumentar el número de épocas mejora el ajuste hasta entrar en una zona de rendimientos decrecientes; en esta implementación pasar de 100 a 300 épocas produjo la mejora más fuerte.",
    "3. El learning rate demasiado chico entrena muy lento, mientras que valores intermedios/altos funcionaron mejor en este problema simple.",
    "4. La arquitectura importa: redes demasiado chicas pueden quedarse cortas y redes más profundas no siempre mejoran si no se ajusta bien el entrenamiento.",
    "5. Refinar la grilla aumenta la cantidad/calidad de datos y también el tiempo de entrenamiento. En este problema la frontera es simple, así que incluso con pocos puntos se logra un desempeño alto.",
    "6. El mini-batch alcanzó alta accuracy en menos épocas, aunque en esta implementación vectorizada simple el tiempo total no siempre fue menor que full-batch por el costo del loop en Python.",
    f"7. ReLU mostró una convergencia muy competitiva: accuracy de evaluación = {relu_metrics['accuracy']:.4f}, con menor loss que la comparación sigmoidea a igual orden de complejidad.",
]

print("\n".join(summary_lines))

with open(RESULTS_DIR / "23_conclusions.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines))
